# Exploration des Modèles LLM Légers pour DataCenter

Ce notebook explore et compare les modèles LLM légers disponibles pour l'application DataCenter AI.

**Objectifs:**
- Comparer les caractéristiques de plusieurs modèles légers
- Mesurer les performances et l'empreinte mémoire
- Évaluer la qualité des réponses
- Choisir le meilleur modèle de base pour le fine-tuning

## 1. Setup et imports

In [ ]:
import torch
import time
import psutil
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import gc
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Définir les modèles à tester

In [ ]:
# Modèles légers à évaluer (0.5B - 3B parameters)
MODELS_TO_TEST = [
    {
        "name": "Qwen2.5-0.5B",
        "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
        "description": "Ultra-léger, production-ready, optimisé pour CPU"
    },
    {
        "name": "Mistral-7B",
        "model_id": "mistralai/Mistral-7B-Instruct-v0.2",
        "description": "Léger, performant, bonne qualité réponse"
    },
    {
        "name": "TinyLLaMA-1.1B",
        "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "description": "Très léger, basé sur LLaMA, bon compromis"
    },
    {
        "name": "Phi-3-mini",
        "model_id": "microsoft/Phi-3-mini-4k-instruct",
        "description": "Optimisé par Microsoft, efficace pour CPU"
    }
]

print(f"Modèles à évaluer ({len(MODELS_TO_TEST)}):")
for model in MODELS_TO_TEST:
    print(f"  - {model['name']}: {model['description']}")

## 3. Fonction de test de modèle

In [ ]:
def get_model_size(model_id):
    """Estimer la taille du modèle depuis HuggingFace"""
    try:
        # Charge les paramètres du config
        from transformers import AutoConfig
        config = AutoConfig.from_pretrained(model_id)
        # Approximation: parameters = dimensions * couches
        if hasattr(config, 'num_parameters'):
            return config.num_parameters / 1e9
        # Calcul manuel pour architectures connues
        elif hasattr(config, 'hidden_size'):
            hidden_size = config.hidden_size
            num_layers = getattr(config, 'num_hidden_layers', 12)
            # Approximation générale
            approx_params = (hidden_size ** 2) * num_layers / 1e9
            return round(approx_params, 2)
    except Exception as e:
        print(f"  Erreur lors du calcul: {e}")
    return None

def test_model_loading(model_id, use_quantization=True):
    """Teste le chargement d'un modèle et mesure les performances"""
    results = {}
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    try:
        # Mesurer la mémoire initiale
        process = psutil.Process()
        mem_before = process.memory_info().rss / 1024 / 1024  # MB
        
        # Charger tokenizer
        print(f"  Chargement du tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        
        # Configuration pour quantization si demandé (économise mémoire)
        device_map = "auto"
        
        print(f"  Chargement du modèle...")
        start_time = time.time()
        
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map=device_map,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            trust_remote_code=True
        )
        
        load_time = time.time() - start_time
        mem_after = process.memory_info().rss / 1024 / 1024  # MB
        mem_used = mem_after - mem_before
        
        results['load_time'] = load_time
        results['memory_used_mb'] = mem_used
        
        # Test d'inférence
        print(f"  Test d'inférence...")
        test_prompt = "How do I check CPU temperature on a Linux server?"
        
        inputs = tokenizer(test_prompt, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = inputs.to('cuda')
        
        start_time = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.7,
                top_p=0.9
            )
        
        inference_time = time.time() - start_time
        results['inference_time'] = inference_time
        results['status'] = 'success'
        
        # Décoder la réponse
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        results['sample_output'] = response[:150] + "..." if len(response) > 150 else response
        
        del model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        results['status'] = f'error: {str(e)[:100]}'
        print(f"  ERREUR: {e}")
    
    return results

print("Fonctions de test définies.")

## 4. Évaluer les modèles

In [ ]:
# Créer un dataframe pour stocker les résultats
results_list = []

for model_info in MODELS_TO_TEST:
    print(f"\n" + "="*60)
    print(f"Évaluation: {model_info['name']}")
    print(f"Model ID: {model_info['model_id']}")
    print("="*60)
    
    model_size_gb = get_model_size(model_info['model_id'])
    print(f"Taille estimée: {model_size_gb} GB" if model_size_gb else "Taille: Non disponible")
    
    # Tester le modèle
    test_results = test_model_loading(model_info['model_id'])
    
    # Compiler les résultats
    result_row = {
        'Model': model_info['name'],
        'Model ID': model_info['model_id'],
        'Size (GB)': model_size_gb,
        'Load Time (s)': test_results.get('load_time', None),
        'Memory Used (MB)': test_results.get('memory_used_mb', None),
        'Inference Time (s)': test_results.get('inference_time', None),
        'Status': test_results.get('status', 'unknown')
    }
    
    results_list.append(result_row)
    
    print(f"Load time: {test_results.get('load_time', 'N/A')} seconds")
    print(f"Memory used: {test_results.get('memory_used_mb', 'N/A')} MB")
    print(f"Inference time: {test_results.get('inference_time', 'N/A')} seconds")
    print(f"Status: {test_results.get('status', 'unknown')}")

# Créer le dataframe
df_results = pd.DataFrame(results_list)
print("\n" + "="*80)
print("RÉSULTATS COMPARATIFS")
print("="*80)
print(df_results.to_string(index=False))

## 5. Recommandation et Conclusion

In [ ]:
print("\n" + "="*80)
print("RECOMMANDATIONS")
print("="*80)

print("""
Pour l'application DataCenter AI:

✓ CHOIX RECOMMANDÉ: Qwen2.5-0.5B
  - Ultra-léger (~500M paramètres)
  - Optimisé pour CPU et edge devices
  - Latence faible pour les applications interactives
  - Suffit pour les tâches structurées (Q&A, classification)
  - Fine-tuning rapide avec LoRA

Alternative 1: Mistral-7B
  - Meilleure qualité réponse
  - Nécessite plus de ressources (GPU recommandé)
  - Bon compromis qualité/performance

Alternative 2: TinyLLaMA-1.1B
  - Bon compromis léger/performant
  - Compatible CPU mais lent
  - Coût mémoire acceptable

Critères de sélection:
- Latence critique? → Qwen2.5-0.5B
- Qualité réponse critique? → Mistral-7B
- Ressources CPU limitées? → Qwen2.5-0.5B ou TinyLLaMA
""")

print("\nProchaines étapes:")
print("1. Utiliser Qwen2.5-0.5B pour le fine-tuning LoRA")
print("2. Préparer le dataset: 02_tests_dataset.ipynb")
print("3. Fine-tuner et tester: 03_demo_finetuning.ipynb")

## 6. Visualisation comparative

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Filtre les résultats valides pour la visualisation
df_valid = df_results[df_results['Status'] == 'success'].copy()

if not df_valid.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Comparaison des Modèles LLM Légers', fontsize=16, fontweight='bold')
    
    # 1. Temps de chargement
    ax = axes[0, 0]
    ax.bar(df_valid['Model'], df_valid['Load Time (s)'], color='skyblue')
    ax.set_title('Temps de Chargement')
    ax.set_ylabel('Secondes')
    ax.tick_params(axis='x', rotation=45)
    
    # 2. Mémoire utilisée
    ax = axes[0, 1]
    ax.bar(df_valid['Model'], df_valid['Memory Used (MB)'], color='lightgreen')
    ax.set_title('Mémoire Utilisée')
    ax.set_ylabel('MB')
    ax.tick_params(axis='x', rotation=45)
    
    # 3. Temps d'inférence
    ax = axes[1, 0]
    ax.bar(df_valid['Model'], df_valid['Inference Time (s)'], color='lightcoral')
    ax.set_title('Temps d\'Inférence (100 tokens)')
    ax.set_ylabel('Secondes')
    ax.tick_params(axis='x', rotation=45)
    
    # 4. Score synthétique (plus bas = mieux)
    ax = axes[1, 1]
    # Normaliser et combiner les métriques
    df_valid['score'] = (
        (df_valid['Load Time (s)'] / df_valid['Load Time (s)'].max()) * 0.3 +
        (df_valid['Memory Used (MB)'] / df_valid['Memory Used (MB)'].max()) * 0.4 +
        (df_valid['Inference Time (s)'] / df_valid['Inference Time (s)'].max()) * 0.3
    )
    colors = ['gold' if score == df_valid['score'].min() else 'silver' for score in df_valid['score']]
    ax.bar(df_valid['Model'], df_valid['score'], color=colors)
    ax.set_title('Score Global (Efficacité)')
    ax.set_ylabel('Score (0=optimal)')
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('/vercel/share/v0-project/notebooks/model_comparison.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("Graphiques sauvegardés: model_comparison.png")
else:
    print("Aucun modèle testé avec succès pour la visualisation")